In [ ]:
# ============================================================
# ANN PDP AND LHS RESPONSE DIAGNOSTICS
# ============================================================
# This block is intended to be placed after the ANN regression
# models and scalers have been trained.
#
# Required objects from previous ANN workflow:
# - df
# - X_COLS
# - mlp_total, scaler_X_total
# - mlp_leak, scaler_X_leak, scaler_y_leak
# - FIGURES_DIR, TABLES_DIR
#
# Purpose:
# - Generate PDP + hexbin diagnostics for ANN-predicted leaked/stored CO2 mass.
# - Generate LHS-based median-scenario response diagnostics for Appendix.
# - Use the original leakage-relevant dataset without oversampling.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import qmc


# ============================================================
# SETTINGS AND INTERPRETABILITY DATASET
# ============================================================

RANDOM_STATE = 42
DISTANCE_FACTOR_M = 20.0

# Use leakage-relevant original dataset: no secure cases, no oversampling.
df_interp = df[df["leak_category"] != "secure"].copy()
df_interp.reset_index(drop=True, inplace=True)

X_interp = df_interp[X_COLS].values

print("\n=== INTERPRETABILITY DATASET FOR PDP AND LHS ===")
print("Dataset shape:", df_interp.shape)
print("\nClass distribution:")
print(df_interp["leak_category"].value_counts())

pretty_feature_map = {
    "PERM_permeability": "Reservoir permeability",
    "PERM_perm_factor_of_failed_well": "Adjacent-well failure factor",
    "PERM_dist_to_well_in_grid": "Injector-to-well distance",
    "PERM_caprock_perm": "Caprock permeability",
    "porosity": "Reservoir porosity",
    "aquifer_porosity": "Aquifer porosity",
    "aquifer_permeability": "Aquifer permeability",
}

axis_label_map = {
    "PERM_dist_to_well_in_grid": "Injector-to-well distance (m)",
    "PERM_permeability": "Reservoir permeability (mD)",
    "PERM_perm_factor_of_failed_well": "Adjacent-well failure factor (-)",
    "PERM_caprock_perm": "Caprock permeability (mD)",
    "porosity": "Reservoir porosity (-)",
    "aquifer_porosity": "Aquifer porosity (-)",
    "aquifer_permeability": "Aquifer permeability (mD)",
}

# Main-text PDP variables for leaked CO2 mass.
# These correspond to the three highest-ranked Morris-like sensitivity controls.
main_leakage_pdp_vars = [
    "PERM_dist_to_well_in_grid",
    "PERM_permeability",
    "porosity",
]

# Supplementary PDP variables for Appendix A.
supplementary_leakage_pdp_vars = [
    "PERM_perm_factor_of_failed_well",
    "PERM_caprock_perm",
]

# Additional variables for LHS diagnostics.
leakage_lhs_vars = [
    "PERM_dist_to_well_in_grid",
    "PERM_permeability",
    "PERM_perm_factor_of_failed_well",
    "PERM_caprock_perm",
    "porosity",
]

stored_lhs_vars = [
    "porosity",
]


# ============================================================
# SAVE INPUT RANGES USED FOR INTERPRETATION
# ============================================================

range_table = pd.DataFrame({
    "Variable": X_COLS,
    "Variable_pretty": [pretty_feature_map.get(col, col) for col in X_COLS],
    "Min": [df_interp[col].min() for col in X_COLS],
    "Max": [df_interp[col].max() for col in X_COLS],
    "Mean": [df_interp[col].mean() for col in X_COLS],
    "Std": [df_interp[col].std() for col in X_COLS],
})

range_table_display = range_table.copy()
range_table_display[["Min", "Max", "Mean", "Std"]] = (
    range_table_display[["Min", "Max", "Mean", "Std"]].round(4)
)

print("\n=== INPUT RANGES USED IN INTERPRETABILITY ANALYSIS ===")
print(range_table_display.to_string(index=False))

range_table.to_csv(
    TABLES_DIR / "Interpretability_InputRanges_exact.csv",
    index=False
)

range_table_display.to_csv(
    TABLES_DIR / "Interpretability_InputRanges_rounded.csv",
    index=False
)


# ============================================================
# PREDICTION WRAPPERS IN PHYSICAL UNITS
# ============================================================

def predict_stored_from_raw(X_raw):
    """
    Predict stored CO2 mass in physical units.
    """
    X_scaled = scaler_X_total.transform(X_raw)
    return mlp_total.predict(X_scaled)


def predict_leaked_from_raw(X_raw):
    """
    Predict leaked CO2 mass in physical units.

    The leaked-mass ANN was trained using a standardized target,
    so predictions are inverse-transformed.
    """
    X_scaled = scaler_X_leak.transform(X_raw)
    y_scaled = mlp_leak.predict(X_scaled)

    y_pred = scaler_y_leak.inverse_transform(
        y_scaled.reshape(-1, 1)
    ).ravel()

    return np.maximum(y_pred, 0.0)


# ============================================================
# PDP WITH HEXBIN DENSITY
# ============================================================

def pdp_hexbin_plot(
    variable_name,
    X_raw,
    feature_names,
    predict_function,
    target_label,
    title_prefix,
    filename_prefix,
    n_grid=60,
    gridsize=35,
    outdir=FIGURES_DIR,
    dpi=600,
    label_map=None,
    distance_factor_m=20.0
):
    """
    Generate a partial dependence plot with observed prediction density.

    The hexbin background shows ANN predictions at original data points.
    The red line shows the partial dependence response.
    The shaded region shows the P10-P90 prediction range at each grid value.
    """
    outdir.mkdir(parents=True, exist_ok=True)

    if variable_name not in feature_names:
        raise ValueError(f"{variable_name} not found in feature_names.")

    feat_idx = feature_names.index(variable_name)
    pretty_name = label_map.get(variable_name, variable_name) if label_map else variable_name

    convert_distance = variable_name == "PERM_dist_to_well_in_grid"

    x_obs_model = X_raw[:, feat_idx].copy()
    y_obs = predict_function(X_raw)

    x_grid_model = np.linspace(
        x_obs_model.min(),
        x_obs_model.max(),
        n_grid
    )

    pdp_mean = []
    pdp_p10 = []
    pdp_p90 = []

    for x_val in x_grid_model:
        X_temp = X_raw.copy()
        X_temp[:, feat_idx] = x_val

        y_temp = predict_function(X_temp)

        pdp_mean.append(np.mean(y_temp))
        pdp_p10.append(np.percentile(y_temp, 10))
        pdp_p90.append(np.percentile(y_temp, 90))

    pdp_mean = np.array(pdp_mean)
    pdp_p10 = np.array(pdp_p10)
    pdp_p90 = np.array(pdp_p90)

    if convert_distance:
        x_obs_plot = x_obs_model * distance_factor_m
        x_grid_plot = x_grid_model * distance_factor_m
        x_label = "Injector-to-well distance (m)"
    else:
        x_obs_plot = x_obs_model
        x_grid_plot = x_grid_model
        x_label = pretty_name

    fig, ax = plt.subplots(figsize=(8, 5.2), dpi=dpi)

    hb = ax.hexbin(
        x_obs_plot,
        y_obs,
        gridsize=gridsize,
        bins="log",
        mincnt=1,
        cmap="Blues",
        linewidths=0.2,
        alpha=0.92
    )

    cbar = fig.colorbar(hb, ax=ax)
    cbar.set_label("log(count)", fontsize=11)
    cbar.ax.tick_params(labelsize=10)

    ax.fill_between(
        x_grid_plot,
        pdp_p10,
        pdp_p90,
        alpha=0.22,
        color="#9ecae1",
        label="P10-P90 range"
    )

    ax.plot(
        x_grid_plot,
        pdp_mean,
        color="crimson",
        linewidth=2.5,
        label="Partial dependence"
    )

    ax.set_xlabel(x_label, fontsize=13)
    ax.set_ylabel(target_label, fontsize=13)
    ax.set_title(f"{title_prefix} — {x_label}", fontsize=14)

    ax.tick_params(axis="both", labelsize=11)
    ax.grid(alpha=0.25, linestyle="--")

    legend = ax.legend(
        loc="best",
        frameon=True,
        fancybox=True,
        framealpha=0.90,
        edgecolor="black",
        fontsize=10
    )

    legend.get_frame().set_linewidth(0.8)
    legend.get_frame().set_facecolor("white")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    safe_name = (
        x_label.replace(" ", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
    )

    output_path = outdir / f"{filename_prefix}_{safe_name}.png"
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"Saved PDP figure: {output_path}")


# ============================================================
# MAIN PDP FIGURES FOR LEAKED CO2 MASS
# ============================================================

for var in main_leakage_pdp_vars:
    pdp_hexbin_plot(
        variable_name=var,
        X_raw=X_interp,
        feature_names=X_COLS,
        predict_function=predict_leaked_from_raw,
        target_label="Predicted leaked CO₂ mass (t)",
        title_prefix="PDP with hexbin density — Leaked CO₂ mass",
        filename_prefix="Figure_10_PDP_LeakedCO2Mass",
        n_grid=60,
        gridsize=35,
        outdir=FIGURES_DIR,
        dpi=600,
        label_map=axis_label_map,
        distance_factor_m=DISTANCE_FACTOR_M
    )


# ============================================================
# SUPPLEMENTARY PDP FIGURES FOR APPENDIX
# ============================================================

for var in supplementary_leakage_pdp_vars:
    pdp_hexbin_plot(
        variable_name=var,
        X_raw=X_interp,
        feature_names=X_COLS,
        predict_function=predict_leaked_from_raw,
        target_label="Predicted leaked CO₂ mass (t)",
        title_prefix="Supplementary PDP — Leaked CO₂ mass",
        filename_prefix="Appendix_A2_PDP_LeakedCO2Mass",
        n_grid=60,
        gridsize=35,
        outdir=FIGURES_DIR,
        dpi=600,
        label_map=axis_label_map,
        distance_factor_m=DISTANCE_FACTOR_M
    )

# Optional stored-mass PDP for Appendix.
for var in stored_lhs_vars:
    pdp_hexbin_plot(
        variable_name=var,
        X_raw=X_interp,
        feature_names=X_COLS,
        predict_function=predict_stored_from_raw,
        target_label="Predicted stored CO₂ mass (t)",
        title_prefix="Supplementary PDP — Stored CO₂ mass",
        filename_prefix="Appendix_A2_PDP_StoredCO2Mass",
        n_grid=60,
        gridsize=35,
        outdir=FIGURES_DIR,
        dpi=600,
        label_map=axis_label_map,
        distance_factor_m=DISTANCE_FACTOR_M
    )


# ============================================================
# LHS-BASED RESPONSE DIAGNOSTICS
# ============================================================

def lhs_response_plot(
    variable_name,
    X_raw,
    feature_names,
    predict_function,
    target_label,
    title_prefix,
    filename_prefix,
    n_points=20,
    outdir=FIGURES_DIR,
    dpi=600,
    label_map=None,
    distance_factor_m=20.0
):
    """
    Generate a one-dimensional LHS-based median-scenario response plot.

    LHS samples are used only for response interpretation with the
    trained ANN models. They are not used for CMG-GEM simulation
    generation or ML model training.
    """
    outdir.mkdir(parents=True, exist_ok=True)

    if variable_name not in feature_names:
        raise ValueError(f"{variable_name} not found in feature_names.")

    feat_idx = feature_names.index(variable_name)
    pretty_name = label_map.get(variable_name, variable_name) if label_map else variable_name

    convert_distance = variable_name == "PERM_dist_to_well_in_grid"

    x_base = np.median(X_raw, axis=0)

    sampler = qmc.LatinHypercube(d=1, seed=RANDOM_STATE)
    u = sampler.random(n=n_points).flatten()

    xmin = X_raw[:, feat_idx].min()
    xmax = X_raw[:, feat_idx].max()

    sampled_vals_model = xmin + u * (xmax - xmin)

    X_lhs = np.tile(x_base, (n_points, 1))
    X_lhs[:, feat_idx] = sampled_vals_model

    y_pred = predict_function(X_lhs)

    sort_idx = np.argsort(sampled_vals_model)

    sampled_vals_model = sampled_vals_model[sort_idx]
    y_pred = y_pred[sort_idx]

    if convert_distance:
        sampled_vals_plot = sampled_vals_model * distance_factor_m
        x_label = "Injector-to-well distance (m)"
    else:
        sampled_vals_plot = sampled_vals_model
        x_label = pretty_name

    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=dpi)

    ax.scatter(
        sampled_vals_plot,
        y_pred,
        color="#4A76FF",
        edgecolor="black",
        s=60,
        alpha=0.85
    )

    ax.plot(
        sampled_vals_plot,
        y_pred,
        color="#4A76FF",
        linewidth=1.5,
        alpha=0.75
    )

    ax.set_xlabel(x_label, fontsize=13)
    ax.set_ylabel(target_label, fontsize=13)
    ax.set_title(f"{title_prefix} — {x_label}", fontsize=14)

    ax.tick_params(axis="both", labelsize=11)
    ax.grid(alpha=0.30, linestyle="--")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    safe_name = (
        x_label.replace(" ", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
    )

    output_path = outdir / f"{filename_prefix}_{safe_name}.png"
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")

    plt.show()
    plt.close(fig)

    print(f"Saved LHS response figure: {output_path}")


for var in leakage_lhs_vars:
    lhs_response_plot(
        variable_name=var,
        X_raw=X_interp,
        feature_names=X_COLS,
        predict_function=predict_leaked_from_raw,
        target_label="Predicted leaked CO₂ mass (t)",
        title_prefix="LHS median-scenario response — Leaked CO₂ mass",
        filename_prefix="Appendix_A3_LHS_LeakedCO2Mass",
        n_points=20,
        outdir=FIGURES_DIR,
        dpi=600,
        label_map=axis_label_map,
        distance_factor_m=DISTANCE_FACTOR_M
    )

for var in stored_lhs_vars:
    lhs_response_plot(
        variable_name=var,
        X_raw=X_interp,
        feature_names=X_COLS,
        predict_function=predict_stored_from_raw,
        target_label="Predicted stored CO₂ mass (t)",
        title_prefix="LHS median-scenario response — Stored CO₂ mass",
        filename_prefix="Appendix_A3_LHS_StoredCO2Mass",
        n_points=20,
        outdir=FIGURES_DIR,
        dpi=600,
        label_map=axis_label_map,
        distance_factor_m=DISTANCE_FACTOR_M
    )


print("\nPDP and LHS response diagnostics completed successfully.")